# Analyses from the master table

Every section below reads **only** `MASTER_PATH` — no joins to the annotation, association,
correlation or genebass files. Each section declares its own parameter block
(`variant_class`, config file, `selected_categories`, `mac`, …) so sections stay independent.

Shared setup is in §0: config loading, the derived-column helper, and the variant filter builder.

Master table built by [`utils/create_master_table.ipynb`](../utils/create_master_table.ipynb).
Grain is one row per `(id, region)`; gene/trait columns (`phenotype`, `loftee_corr_dir`, …)
are already attached to every variant row.

## 0. Shared setup

In [ ]:
import yaml
import numpy as np
import polars as pl

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import env_override, fetch_hf_data
CONFIG_DIR  = str(REPO_ROOT / 'configs')
MASTER_PATH  = env_override('MASTER_PATH', fetch_hf_data('genebass_annotated.parquet', REPO_ROOT))
FIG_DIR     = env_override('FIG_DIR', '../../paper_figures')

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT))
from utils.variant_filtering import (
    load_config, load_variant_class, scan_variants, appv_of, pick_annos, filter_covered, derived_schema,
    env_override,
)

_THEME = theme_minimal() + theme(
    axis_text=element_text(size=11, lineheight=1.4),
    axis_title=element_text(size=12),
    legend_text=element_text(size=12),
    legend_title=element_text(size=12),
    plot_background=element_rect(fill='white', color='white'),
)

MASTER_COLS = pl.scan_parquet(MASTER_PATH).collect_schema().names()
DERIVED_COLS = derived_schema(MASTER_PATH)
print(f'master table: {len(MASTER_COLS)} cols -> {len(DERIVED_COLS)} after add_derived()')


## 1. Average z-score by category

For binary annotations: mean direction-corrected genebass beta of the variants carrying each
flag, averaged over gene-trait pairs. Reproduces `avg_zscore_categories.ipynb`.

Note this section keeps indels (`only_snps=False`) and applies `variant_length <= 50`.

In [ ]:
# --- parameters (env_override(NAME, default) -- set UKBBGYM_NAME to override without editing)
C2_variant_class       = env_override('VARIANT_CLASS', 'missense')
C2_config              = env_override('MEAN_PHENO_CONFIG_FILE', 'config_categories.yaml')
C2_selected_categories = env_override('MEAN_PHENO_CATEGORIES', ['protein_domains'], 'list')
C2_mac                 = env_override('MAC', 20, int)
C2_only_snps           = env_override('ONLY_SNPS', False, bool)
C2_only_clinvar        = env_override('ONLY_CLINVAR', False, bool)
C2_exclude_clinvar     = env_override('EXCLUDE_CLINVAR', False, bool)
C2_max_variant_length  = env_override('MAX_VARIANT_LENGTH', 50, int)
C2_ci_factor           = env_override('CI_FACTOR', 1.96, float)

In [ ]:
c2_cfg, c2_all = load_config(CONFIG_DIR, C2_config)
c2_vc  = load_variant_class(CONFIG_DIR, C2_variant_class)
c2_lf  = scan_variants(MASTER_PATH, c2_vc, only_snps=C2_only_snps, only_clinvar=C2_only_clinvar,
                       exclude_clinvar=C2_exclude_clinvar,
                       max_variant_length=C2_max_variant_length)
c2_annos = pick_annos(c2_cfg, c2_all, C2_selected_categories, DERIVED_COLS)

# Binary annotations: keep only the variants carrying the flag
c2_melted = (c2_lf
    .select(set(['id', 'region']) | set(c2_annos))
    .unpivot(index=['id', 'region'], on=c2_annos,
             variable_name='annotation', value_name='annotation_score')
    .filter(pl.col('annotation_score') == 1)
    .with_columns(pl.col('annotation_score').cast(pl.Float32), pl.col('region').cast(pl.Utf8))
    .join(c2_cfg.select(['annotation', 'category', 'annotation_dir']).lazy(), on='annotation', how='left')
    .filter(pl.col('category').is_in(C2_selected_categories))
    .unique()
    .with_columns(annotation_score_dircor=pl.when(pl.col('annotation_dir') != 1)
                  .then((pl.col('annotation_score') - 1).abs())
                  .otherwise(pl.col('annotation_score'))))

all_pheno_df = (
    appv_of(c2_lf, C2_mac)
    .join(c2_melted, on=['id', 'region'], how='inner')
    .join(c2_lf.select(['region', 'phenotype', 'loftee_corr_dir']).unique(),
          on=['region', 'phenotype'], how='inner')
    .with_columns(mean_pheno_value_dircor=pl.col('mean_pheno_value') * pl.col('loftee_corr_dir'))
    .group_by(['annotation', 'region', 'phenotype'])
    .agg(n_vars=pl.col('id').n_unique(),
         mean_pheno_assoc=pl.col('mean_pheno_value_dircor').mean())
    .collect(engine='streaming'))

avg_pheno_df = (all_pheno_df.group_by('annotation')
    .agg(n_gene_trait_assoc=pl.col('region').n_unique(),
         mean_pheno=pl.col('mean_pheno_assoc').mean(),
         se_pheno=pl.col('mean_pheno_assoc').std() / pl.col('region').n_unique().sqrt())
    .with_columns(ci_low_pheno=pl.col('mean_pheno') - C2_ci_factor * pl.col('se_pheno'),
                  ci_high_pheno=pl.col('mean_pheno') + C2_ci_factor * pl.col('se_pheno')))
avg_pheno_df

In [ ]:
c2_plot_df = (avg_pheno_df
    .join(c2_cfg.drop('category').unique(), on='annotation')
    .with_columns(n_label=pl.col('n_gene_trait_assoc').cast(pl.Utf8) + pl.lit(' genes'))
    .drop_nans())
c2_order = c2_plot_df.sort('mean_pheno')['label']
c2_plot_df = c2_plot_df.with_columns(pl.col('label').cast(pl.Enum(c2_order)))

c2_title = c2_vc['x_label']
if C2_exclude_clinvar:
    c2_title += ' (no ClinVar)'
elif C2_only_clinvar:
    c2_title += ' (only ClinVar)'
if C2_only_snps:
    c2_title += ' (SNPs)'

c2_plot = (
    ggplot(c2_plot_df, aes(x='label', y='mean_pheno'))
    + geom_hline(yintercept=0, linetype='dashed', color='grey')
    + geom_point(size=3)
    + geom_errorbar(aes(ymin='ci_low_pheno', ymax='ci_high_pheno'), width=0.2)
    + geom_text(aes(label='n_label'), size=11, nudge_x=0.25, ha='center', va='bottom')
    + coord_flip()
    + labs(
        x='',
        y='Mean phenotype across carriers ± 1.96 × s.e.m.\n(direction-corrected)'
    )
    + _THEME 
    + theme(
        figure_size=(7, c2_plot_df.height * 0.7 + 1),
        axis_text=element_text(size=12.5),
        axis_title_x=element_text(size=13, lineheight=1.4),
    )
)
c2_plot.save(f'{FIG_DIR}/F4_{C2_selected_categories[0]}_mean_phenotype.svg', dpi=200, verbose=False)
c2_plot

In [ ]:
c2_num_df = (all_pheno_df
    .join(c2_cfg.drop('category').unique(), on='annotation')
    .join(all_pheno_df.group_by('annotation').agg(n_genes=pl.col('region').n_unique()), on='annotation')
    .with_columns(n_label=pl.col('label') + '\n(genes=' + pl.col('n_genes').cast(pl.Utf8) + ')')
    .drop_nans())
_map = dict(c2_num_df.select('label', 'n_label').unique().iter_rows())

c2_num_plot = (
    ggplot(c2_num_df, aes(x='n_label', y='n_vars'))
    + geom_boxplot(width=0.5)
    + geom_jitter(width=0.1, size=1, alpha=0.1)
    + scale_x_discrete(limits=[_map[l] for l in c2_order if l in _map])
    + coord_flip() + scale_y_log10()
    + labs(x='', y='Variants per gene')
    + _THEME + theme(figure_size=(6, c2_num_df['annotation'].n_unique() * 0.6 + 1),
                     panel_grid_major=element_line(color='lightgray', size=0.4),
                     panel_grid_minor=element_line(color='lightgray', size=0.2))
)
c2_num_plot.save(f'{FIG_DIR}/master_{C2_selected_categories[0]}_variants_per_gene.svg', dpi=200, verbose=False)
c2_num_plot